# Day 25 — **Abstract Base Classes: Agent Interface**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

Once you have more than one agent — an invoice agent, a fraud agent, a KYC agent — an orchestrator needs to treat them **uniformly**: `agent.run()`, `agent.health_check()`, regardless of type. An **Abstract Base Class** defines that contract and makes Python *refuse to instantiate* any agent that doesn't implement it. The interface stops being a hopeful convention and becomes an enforced one.

Today:
1. `BaseAgent(ABC)` with `@abstractmethod` — `run`, `validate`, `health_check`.
2. Two concrete agents: `InvoiceAgent`, `FraudAgent`.
3. `@property` in the contract, and what happens when you skip a method.

> Pure stdlib (`abc`). No keys.

## 1. Define the contract with `ABC`

Subclass `ABC` and mark required methods `@abstractmethod`. Python then **blocks instantiation** of any subclass that hasn't overridden *every* abstract member — the error fires at construction, not deep in production when the missing method is finally called.

In [1]:
from abc import ABC, abstractmethod

class BaseAgent(ABC):
    @property
    @abstractmethod
    def name(self) -> str:
        '''Stable identifier used in logs and routing.'''

    @abstractmethod
    def validate(self, payload: dict) -> bool:
        '''Return True if payload has the fields this agent needs.'''

    @abstractmethod
    def run(self, payload: dict) -> dict:
        '''Do the work. Must call validate() first.'''

    @abstractmethod
    def health_check(self) -> bool:
        '''Cheap readiness probe for the orchestrator / load balancer.'''

print("BaseAgent defined with 4 abstract members:",
      sorted(BaseAgent.__abstractmethods__))

BaseAgent defined with 4 abstract members: ['health_check', 'name', 'run', 'validate']


☝ `__abstractmethods__` lists what a subclass **must** implement: `name`, `validate`, `run`, `health_check`. `@property` stacked over `@abstractmethod` makes `name` a required *property*, not a method — callers write `agent.name`, not `agent.name()`. The contract is now machine-enforced.

## 2. Two concrete agents

`InvoiceAgent` and `FraudAgent` each implement all four members. They share the interface but differ entirely in logic — which is the point: the orchestrator doesn't care which is which.

In [2]:
class InvoiceAgent(BaseAgent):
    @property
    def name(self) -> str:
        return "invoice-agent"

    def validate(self, payload: dict) -> bool:
        return {"invoice_id", "amount"} <= payload.keys()

    def run(self, payload: dict) -> dict:
        if not self.validate(payload):
            return {"ok": False, "error": "missing invoice_id/amount"}
        approved = payload["amount"] <= 10_000        # auto-approve small invoices
        return {"ok": True, "agent": self.name, "approved": approved}

    def health_check(self) -> bool:
        return True


class FraudAgent(BaseAgent):
    @property
    def name(self) -> str:
        return "fraud-agent"

    def validate(self, payload: dict) -> bool:
        return {"txn_id", "amount", "country"} <= payload.keys()

    def run(self, payload: dict) -> dict:
        if not self.validate(payload):
            return {"ok": False, "error": "missing txn fields"}
        risky = payload["amount"] > 5_000 and payload["country"] != "UK"
        return {"ok": True, "agent": self.name, "flag": "review" if risky else "clear"}

    def health_check(self) -> bool:
        return True


agents = [InvoiceAgent(), FraudAgent()]
print("instantiated:", [a.name for a in agents])

instantiated: ['invoice-agent', 'fraud-agent']


☝ Both instantiate cleanly because both satisfy the full contract. Note `a.name` (property, no parens). Two unrelated implementations now share one shape — the precondition for polymorphic orchestration.

## 3. Polymorphism — the orchestrator treats them uniformly

Because both are `BaseAgent`, an orchestrator loops over them identically: probe health, then run. It never branches on concrete type — new agent types drop in without touching this code.

In [3]:
def dispatch(agents, payload):
    results = []
    for a in agents:
        if not a.health_check():          # skip unhealthy agents
            results.append((a.name, "SKIPPED: unhealthy")); continue
        if a.validate(payload):           # only agents that accept this payload
            results.append((a.name, a.run(payload)))
    return results

invoice_payload = {"invoice_id": "INV-9", "amount": 8500}
fraud_payload   = {"txn_id": "T-3", "amount": 9000, "country": "RO"}

for p in (invoice_payload, fraud_payload):
    print("payload:", p)
    for name, out in dispatch(agents, p):
        print("   ", name, "->", out)
    print()

payload: {'invoice_id': 'INV-9', 'amount': 8500}
    invoice-agent -> {'ok': True, 'agent': 'invoice-agent', 'approved': True}

payload: {'txn_id': 'T-3', 'amount': 9000, 'country': 'RO'}
    fraud-agent -> {'ok': True, 'agent': 'fraud-agent', 'flag': 'review'}



☝ Same `dispatch` loop routes each payload to whichever agent `validate`s it — invoice fields go to the invoice agent, txn fields to the fraud agent — with no `if isinstance(...)` anywhere. Add a `KYCAgent(BaseAgent)` tomorrow and `dispatch` picks it up unchanged. That's the payoff of coding to the abstract type.

## 4. Skip a method → Python refuses to construct it

The enforcement: a subclass missing even one abstract member **can't be instantiated**. The error is immediate and names the gap — you find out at construction, not when a caller hits the missing method in production.

In [4]:
class BrokenAgent(BaseAgent):
    @property
    def name(self) -> str:
        return "broken"
    def run(self, payload): return {}
    # forgot: validate(), health_check()

try:
    BrokenAgent()
except TypeError as e:
    print("TypeError:", e)

TypeError: Can't instantiate abstract class BrokenAgent without an implementation for abstract methods 'health_check', 'validate'


☝ `Can't instantiate abstract class BrokenAgent ... abstract methods health_check, validate` — Python blocks it at `BrokenAgent()`. A plain base class with `raise NotImplementedError` would only fail *when the method is called*, possibly in production. ABCs move that failure to the earliest possible moment.

## 5. Recap — enforced agent contracts

| Piece | Effect |
|---|---|
| `class BaseAgent(ABC)` | declares an interface |
| `@abstractmethod` | subclass **must** override; else uninstantiable |
| `@property` + `@abstractmethod` | required attribute-style member (`agent.name`) |
| `__abstractmethods__` | the enforced checklist |
| polymorphic `dispatch` | orchestrator codes to the type, not the class |

ABCs turn *"all agents should have a `run` method"* from a code-review comment into a language guarantee — the foundation for the **agentic architecture patterns** 